# PinVuTru Cafe POS · Fine-tune Daylight AI (`Qwen 2.5 3B/7B` + `BF16 LoRA` trên Colab Pro A100)

Notebook này huấn luyện mô hình trợ lý tiếng Việt gọi Tool Calling chuẩn khớp 100% với `AI/schemas/tool_schemas.json` của hệ thống **PinVuTru Cafe POS**, tận dụng **NVIDIA A100 40GB (`BF16` + `FlashAttention-2`)** và xuất trực tiếp file **`GGUF (Q4_K_M)`** về Google Drive để chạy local trên **Ollama ổ `D:\AI_Models\ollama`**.

### Trước khi chạy:
1. Chọn Runtime → Change runtime type → **A100 GPU** (chỉ bật A100 lúc chạy thật để tiết kiệm Compute Units).
2. Tải 3 file từ repo lên Google Drive thư mục `/MyDrive/CafePos_AI/`:
   - `AI/schemas/tool_schemas.json`
   - `AI/data/processed/train_tool_calling.jsonl`
   - `AI/data/processed/val_tool_calling.jsonl`

In [ ]:
# 1. Kiểm tra GPU & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Cài Unsloth chuẩn cho Colab GPU (không ghi đè torch CUDA sẵn có)
!pip install --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
# 3. Cấu hình thực nghiệm (đổi MODEL_SIZE = '3b' hoặc '7b' để làm Ablation Study)
import os, json
from pathlib import Path

MODEL_SIZE = '3b'  # '3b' (chạy 100% GPU trên Quadro T2000 4GB) hoặc '7b' (so sánh Ablation)
DRIVE_DIR = Path('/content/drive/MyDrive/CafePos_AI')
SCHEMA_PATH = DRIVE_DIR / 'tool_schemas.json'
TRAIN_PATH = DRIVE_DIR / 'train_tool_calling.jsonl'
VAL_PATH = DRIVE_DIR / 'val_tool_calling.jsonl'
OUT_DIR = DRIVE_DIR / f'outputs_qwen25_{MODEL_SIZE}_lora'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct' if MODEL_SIZE == '3b' else 'Qwen/Qwen2.5-7B-Instruct'
print('Base model:', BASE_MODEL)
print('Output dir:', OUT_DIR)

In [ ]:
# 4. Tải Model (tự động chọn BF16 LoRA trên A100/L4 hoặc QLoRA 4-bit trên T4)
from unsloth import FastLanguageModel
import torch

IS_BF16 = torch.cuda.is_bf16_supported()
print('GPU:', torch.cuda.get_device_name(0), '| BF16 supported:', IS_BF16)

max_seq_length = 4096 if IS_BF16 else 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16 if IS_BF16 else None,
    load_in_4bit=not IS_BF16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

In [ ]:
# 5. Chuẩn bị Dataset theo đúng Qwen 2.5 Tool Calling Chat Template
from datasets import Dataset

schema = json.loads(SCHEMA_PATH.read_text(encoding='utf-8'))
tools = schema['tools']

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def format_row(row):
    text = tokenizer.apply_chat_template(
        row['messages'],
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_ds = Dataset.from_list(load_jsonl(TRAIN_PATH)).map(format_row)
val_ds = Dataset.from_list(load_jsonl(VAL_PATH)).map(format_row)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print('--- SAMPLE PREVIEW ---')
print(train_ds[0]['text'][:700])

In [ ]:
# 6. Huấn luyện SFT với BF16 trên A100 (~2-5 phút)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=max_seq_length,
        per_device_train_batch_size=8 if IS_BF16 else 4,
        gradient_accumulation_steps=2 if IS_BF16 else 4,
        warmup_ratio=0.08,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=IS_BF16,
        fp16=not IS_BF16,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=25,
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=str(OUT_DIR / 'checkpoints'),
        report_to='none',
    ),
)

train_result = trainer.train()
eval_result = trainer.evaluate()
print('Train metrics:', train_result.metrics)
print('Eval metrics:', eval_result)

In [ ]:
# 7. Lưu log loss để đưa vào báo cáo Docs/QUY-TRINH-AI-VA-DANH-GIA.md
history_path = OUT_DIR / 'training_history.json'
history_path.write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
print('Saved training history to:', history_path)

In [ ]:
# 8. Xuất trực tiếp sang GGUF (Q4_K_M) lưu lên Google Drive để chạy Ollama ổ D:\
gguf_dir = OUT_DIR / 'gguf'
model.save_pretrained_gguf(
    str(gguf_dir),
    tokenizer,
    quantization_method='q4_k_m',
)
print('HOÀN TẤT! Tải file *-q4_k_m.gguf về D:\\AI_Models\\ollama\\custom\\cafepos-qwen-lora-q4_k_m.gguf')